# Memorization: recurrent vs feedforward delays

Initialization follows `experiments/compare_mem_ziyad.py`, with configuration from
`configs/perf_MEM.py`. Compare **axonal, synaptic and hybrid** delays on each pathway.
All six models use the same generated samples, labels and batch order, with matched
initial connection weights and base delays from one shared parameter generator.
Accuracy is measured on the training set only.

On Kaggle, enable a GPU accelerator and Internet for the first setup, then run all cells.
The setup fetches only the library source from the `toi_task` branch of your GitHub fork; an existing checkout
(or a checkout attached as input) can be used by setting `REPO_DIR`.
The selected revision must contain the six comparison networks and `spike_memorization.py`.

Edit the **Configuration** cell for each experiment, then rerun that cell and the
**Run comparison** and **Plot comparison** cells. Each run gets a new output directory;
there is no need to clone again. Training and matching helpers are also editable here.
No experiment script or Python configuration file is imported.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ON_KAGGLE = Path('/kaggle/working').is_dir()
WORK_DIR = Path('/kaggle/working') if ON_KAGGLE else Path.cwd()
REPO_URL = 'https://github.com/AlbatorZ/DelRec.git'
REPO_REF = 'toi_task'  # This branch contains the MEM comparison networks.
# For a Kaggle input checkout, replace with its /kaggle/input/... path.
REPO_DIR = WORK_DIR / 'DelRec'
INSTALL_DEPENDENCIES = ON_KAGGLE

# Reuse this repository when running locally from its root or notebooks/ directory.
if not ON_KAGGLE:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src/delrec/networks.py').is_file():
            REPO_DIR = candidate
            break

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'sparse-checkout', 'set', 'src'], check=True)
if not (REPO_DIR / 'src/delrec/datasets/spike_memorization.py').is_file():
    raise FileNotFoundError(f'Set REPO_DIR to a checkout with the MEM library: {REPO_DIR}')

if INSTALL_DEPENDENCIES:
    # Keep Kaggle's installed CUDA-enabled PyTorch. Install only the extras used here.
    # SpikingJelly is pinned to the same source revision as requirements.txt.
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'cloudpickle', 'DCLS==0.1.1', 'numpy', 'matplotlib', 'prettytable',
                    'spikingjelly @ git+https://github.com/fangwei123456/spikingjelly.git@d4fee3a1715bf42ce15f52fde1b1d4709d7ea25e'],
                   check=True)

sys.path.insert(0, str(REPO_DIR / 'src'))
os.environ.setdefault('MPLCONFIGDIR', str(WORK_DIR / '.matplotlib'))
print(f'Library: {REPO_DIR / "src"}')

In [ ]:
from copy import deepcopy
from datetime import datetime
import csv
import json

import matplotlib.pyplot as plt
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader
from spikingjelly.activation_based import neuron, surrogate

from IPython.display import Markdown, display
from delrec.training.run_recap import save_training_recap
from delrec.training.delay_diagnostics import DelayDiagnostics
from delrec import networks
from delrec.datasets.spike_memorization import SpikeMemorization
from delrec.delay_layers import axonal_recdel, vanilla_recurrent
from delrec.networks import dcls_module, learned_delay_parameter
from delrec.utils import reset_states, seed_everything

print(f'PyTorch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}')

## Configuration

These are the current `perf_MEM.py` defaults (256 samples, 16 classes, 200 epochs).
`model` is selected automatically for each of the six runs. Keep `dataset_seed` fixed
when comparing settings. Change values **inside the class** so derived padding and
position bounds are recalculated when this cell runs. For a quick check, use
`epochs = 2`, `num_samples = 16`, and `hidden_layers = [8]`.

`DEVICE = "auto"` uses CUDA if available, otherwise CPU. Results go to a fresh folder
under `OUTPUT_ROOT`. The setup retains the environment's PyTorch version; this is not
a recreation of every dependency pin in the repository's reference environment.

Learning rates are selected from the actual network layers. All recurrent models
(axonal, synaptic, hybrid) use `lr_w_recurrent` / `lr_positions_recurrent`; all
feedforward models use `lr_w_feedforward` / `lr_positions_feedforward`.
These are rates for the whole model, including its input and output projections.
The effective rates are saved as `lr_w` and `lr_positions` in each run's config.
After editing rates, rerun Configuration and Run comparison (and the training
helper cell once if updating an already-open notebook to this version).

Set `delay_diagnostics_every = X` to save and display per-layer histograms at
epoch 0, every X completed epochs, and the final epoch. Snapshots contain learned
positions, effective delays (including hybrid offsets), `|gradient|` before
gradient clipping, and actual `|delay change|` after AdamW and clamping. Update
distributions pool all minibatches in that epoch; epoch 0 has no updates yet.
DCLS positions increase toward the present, so effective delays reverse their sign.
PNG plots, signed raw arrays (NPZ), and statistics (JSON: zero fraction, mean,
spread, magnitude, missing gradients) are saved in each model’s `delay_diagnostics/`.
The initial and final snapshots are always included even when X exceeds epochs.

### Two GPUs

`PARALLEL_GPUS = "auto"` runs feedforward models sequentially on GPU 0 and
recurrent models sequentially on GPU 1 **at the same time** when two CUDA GPUs
are available. Otherwise it uses the usual sequential CPU/single-GPU execution.
Set `True` to require two GPUs, or `False` to force sequential execution.
`DEVICE = "cpu"` keeps automatic mode on CPU.

Workers use separate Python processes and receive the current notebook helpers
and configuration via `cloudpickle` (installed by Setup). Both reconstruct the
same matched initialization on CPU. Each worker sees only its assigned GPU;
`assigned_device` in saved configs and `execution.json` record notebook GPU indices.
Worker-local `device` is therefore `cuda:0` for both isolated processes.
Progress from both workers appears below the run cell and is saved in
`feedforward.log` / `recurrent.log`. Diagnostic PNG/NPZ/JSON files are saved as usual;
parallel workers do not display inline diagnostic plots. The final comparison
plot still combines all six runs. Interrupting the cell stops both workers.

After updating an open notebook, rerun Setup, Configuration, the training helpers,
and Match the six models before Run comparison.

Each model also saves and displays `delay_diagnostics/hidden_delay_evolution.png`: 
a heatmap of hidden-neuron delays at initialization and every completed epoch.
Colors show outgoing delays: the learned axonal part only for hybrid models
(excluding fixed synaptic offsets), and the mean over targets (and kernels)
for synaptic models. Input neurons are excluded; the final feedforward projection
is included because its sources are the last hidden neurons. Multiple hidden
layers are stacked in labeled row blocks. Raw values and row metadata are saved
as NPZ and JSON alongside the figure.


In [ ]:
class Config:
    dataset = "MEM"
    model = "SNN_recurrent_and_feedforward_delays"
    seed = 0
    dataset_seed = 0
    task_type = "temporal"
    num_samples = 256
    input_size = 16
    time_window = 32
    output_size = 4
    input_gain = 1.0
    hidden_layers = [64]
    epochs = 100
    delay_diagnostics_every = 150  # initial, every X completed epochs, and final
    batch_size = 64
    num_workers = 0
    cpu_threads = 1
    readout = "mean"  # mean, sum, or last temporal output

    bias = False
    use_batch_norm = False
    feedforward_dropout_rate = 0.0
    recurrent_dropout_rate = 0.0
    init_ff_weights = "default"
    init_dcls_weights = "default"
    no_delay_in_first_layer = True
    no_delay_in_last_layer = False
    no_recurrence_in_last_layer = False  # remove the last hidden recurrence
    no_recurrent_delays = False  # keep recurrence, fix delays to zero

    neuron_module = neuron.LIFNode
    surrogate_function = surrogate.ATan(alpha=2.0)
    tau = 2.0
    decay_input = False
    v_threshold = 1.0
    v_reset = 0.0
    detach_reset = False
    step_mode = "m"
    backend = "torch"
    store_v_seq = False

    init_rec_weights = "orthogonal"
    rec_delay_init_gain = 0.5
    use_rec_bias = False
    init_rec_delay = "uniform"
    init_recdel_offset = 0.0
    max_rec_delay = 3
    delay_std_init = 2.0
    use_sig_p = False
    sigma_init = 0.0
    sigma_decay = 0.95

    #Hybrid delays configuration
    hybrid_max_synaptic_delay = 2  # fixed integer offsets in [0, 4] on both pathways
    hybrid_delay_seed = 123  # independent of dataset and model seed
    round_delays = False
    round_pos_each_epoch = False

    DCLSversion = "gauss"
    kernel_count = 1
    max_feedforward_delay = 9
    left_padding = max_feedforward_delay - 1
    right_padding = 0  # causal, length-preserving convolution
    init_pos_a = -(max_feedforward_delay // 2)
    init_pos_b = max_feedforward_delay // 2
    siginit = 1.0  # scheduled to 0.23 in the first half of training

    # Rates apply to all weights/delays in the selected network family.
    lr_w_recurrent = 0.005
    lr_positions_recurrent = 0.08
    lr_w_feedforward = 0.005
    lr_positions_feedforward = 0.08
    # Legacy fallback; make_optimizer records the selected effective rates here.
    lr_w = 0.005
    lr_positions = 0.08
    weight_decay = 0.0
    grad_clip = 1.0


config = Config()
DEVICE = 'auto'  # auto, cuda, or cpu
PARALLEL_GPUS = 'auto'  # auto: use 2 GPUs if available; True: require 2; False: sequential
OUTPUT_ROOT = WORK_DIR / 'exp' / 'MEM' / 'delay_location_comparison'

## Training helpers

AdamW uses separate learning rates for weights and delays, with cosine scheduling.
DCLS widths and recurrent smoothing follow the original schedules. Every epoch ends
with a measurement on the entire training set, retaining fractional delays.
These helpers are copied into the notebook so they can be edited independently.

In [ ]:
def get_dcls_sigma_for_epoch(config, epoch: int):

    if getattr(config, "DCLSversion", None) != "gauss":
        return 0.23

    total_epochs = max(1, config.epochs)

    decay_horizon = max(1, total_epochs // 2)

    sigma_min = 0.23
    if epoch >= decay_horizon:
        return sigma_min

    if config.siginit <= sigma_min:
        return sigma_min

    alpha = (sigma_min / float(config.siginit)) ** (1.0 / decay_horizon)
    sigma = float(config.siginit) * (alpha ** epoch)
    return max(sigma, sigma_min)


def make_optimizer(model, config):
    # Classify actual layers, including synaptic/hybrid subclasses and vanilla
    # recurrence. Combined recurrent + feedforward delay networks use rec rates.
    family = ('recurrent' if any(isinstance(m, (axonal_recdel, vanilla_recurrent))
                                for m in model.modules()) else 'feedforward')
    # Preserve effective values in the config saved alongside each checkpoint.
    config.lr_w = getattr(config, f'lr_w_{family}', config.lr_w)
    config.lr_positions = getattr(config, f'lr_positions_{family}', config.lr_positions)
    positions = []
    for module in model.modules():
        if isinstance(module, axonal_recdel):
            positions.append(learned_delay_parameter(module, 'recurrent_delays'))
            if hasattr(module, "p_spread"):
                positions.append(module.p_spread)
        elif isinstance(module, dcls_module):
            positions.append(learned_delay_parameter(module, 'P'))
            # Width is scheduled explicitly, rather than optimized.
            if config.DCLSversion == "gauss":
                module.SIG.requires_grad_(False)
    position_ids = {id(p) for p in positions}
    weights = [p for p in model.parameters() if p.requires_grad and id(p) not in position_ids]
    return torch.optim.AdamW([
        {"params": weights, "lr": config.lr_w, "weight_decay": config.weight_decay},
        {"params": [p for p in positions if p.requires_grad],
         "lr": config.lr_positions, "weight_decay": 0.0},
    ])


def set_epoch(model, config, epoch):
    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, axonal_recdel):
                module.update_sigma(epoch)
            elif isinstance(module, dcls_module) and config.DCLSversion == "gauss":
                module.SIG.fill_(get_dcls_sigma_for_epoch(config, epoch))


def run_epoch(loader, model, device, config, optimizer=None, diagnostics=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for inputs, labels in loader:
            inputs = inputs.permute(1, 0, 2).contiguous().to(device)
            labels = labels.to(device)
            reset_states(model)
            if training:
                optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            if config.readout == "mean":
                logits = outputs.mean(0)
            elif config.readout == "sum":
                logits = outputs.sum(0)
            elif config.readout == "last":
                logits = outputs[-1]
            else:
                raise ValueError(f"Unknown readout: {config.readout}")
            loss = F.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite memorization loss")
            if training:
                loss.backward()
                if diagnostics is not None:
                    diagnostics.before_step()
                if config.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                optimizer.step()
                if hasattr(model, "clamp_delays"):
                    model.clamp_delays()
                if diagnostics is not None:
                    diagnostics.after_step()
            total_loss += loss.item() * labels.numel()
            correct += (logits.argmax(1) == labels).sum().item()
            count += labels.numel()
    reset_states(model)
    return {"loss": total_loss / count, "accuracy_percent": 100.0 * correct / count,
            "correct": correct, "num_samples": count}


In [ ]:
def plot_results(history, final, config, run_dir):
    fig, (loss_ax, acc_ax) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    loss_ax.plot([r["epoch"] for r in history], [r["loss"] for r in history])
    loss_ax.set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Training set (after each epoch)")
    loss_ax.grid(alpha=0.25)
    accuracy = final["accuracy_percent"]
    acc_ax.bar(["Final training accuracy"], [accuracy], width=0.45)
    acc_ax.set(ylabel="Accuracy (%)", ylim=(0, 110))
    acc_ax.axhline(100 / config.output_size, color="gray", linestyle="--", label="Uniform random guess")
    acc_ax.text(0, accuracy + 2, f"{accuracy:.2f}% ({final['correct']}/{final['num_samples']})", ha="center")
    acc_ax.legend(loc="lower right")
    fig.suptitle(f"{config.model}\n{config.task_type}, {config.num_samples} samples, seed {config.seed}")
    fig.savefig(run_dir / "training_summary.png", dpi=180)
    fig.savefig(run_dir / "training_summary.pdf")
    plt.close(fig)


def run(config, device, out, model, show_diagnostics=True):
    """Run one complete experiment, optionally with a preinitialized model."""
    torch.set_num_threads(config.cpu_threads)
    seed_everything(config.seed, is_cuda=torch.cuda.is_available())
    dataset = SpikeMemorization(config.num_samples, config.input_size, config.time_window,
                               config.output_size, config.dataset_seed, config.input_gain, config.task_type)
    loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(config.seed), num_workers=config.num_workers)
    # Same dataset, deterministic order: this is training-set measurement, not a split.
    measure_loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=False,
                                num_workers=config.num_workers)
    model = model.to(device)
    # Support existing checkouts: DCLS caches from the CPU matching probe are
    # ordinary tensors and are not moved by .to(device). Rebuild them on-device.
    for module in model.modules():
        if isinstance(module, dcls_module):
            module.DCK.IDX = None
            module.DCK.lim = None
    optimizer = make_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
    run_dir = Path(out)
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "config.json").exists():
        raise FileExistsError(f"Run already exists: {run_dir}")
    settings = {k: getattr(config, k) for k in dir(config)
                if not k.startswith("_") and not callable(getattr(config, k))}
    settings["device"] = str(device)
    settings["surrogate_function"] = repr(config.surrogate_function)
    (run_dir / "config.json").write_text(json.dumps(settings, indent=2))
    torch.save({"inputs": dataset.inputs, "labels": dataset.labels}, run_dir / "dataset.pt")
    print(f"Device: {device}; model: {config.model}; output: {run_dir}", flush=True)
    diagnostics = DelayDiagnostics(model, run_dir,
                                   every=getattr(config, "delay_diagnostics_every", 10),
                                   show=show_diagnostics)
    diagnostics.snapshot()
    history = []
    with (run_dir / "train_res.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["epoch", "loss", "accuracy_percent", "online_loss"])
        writer.writeheader()
        for epoch in range(config.epochs):
            set_epoch(model, config, epoch)
            diagnostics.begin_epoch(epoch + 1, config.epochs)
            online = run_epoch(loader, model, device, config, optimizer, diagnostics=diagnostics)
            diagnostics.snapshot()
            # Preserve fractional delays and the training-time smoothing for measurement.
            final = run_epoch(measure_loader, model, device, config)
            row = {"epoch": epoch + 1, "loss": final["loss"],
                   "accuracy_percent": final["accuracy_percent"], "online_loss": online["loss"]}
            history.append(row)
            writer.writerow(row)
            stream.flush()
            scheduler.step()
            if epoch == 0 or (epoch + 1) % 10 == 0 or epoch + 1 == config.epochs:
                print(f"Epoch {epoch + 1}/{config.epochs}: loss={final['loss']:.6f}, "
                      f"training accuracy={final['accuracy_percent']:.2f}%", flush=True)
    final.update(epoch=config.epochs, model=config.model, seed=config.seed,
                 dataset_seed=config.dataset_seed, task_type=config.task_type,
                 trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad))
    (run_dir / "final_train.json").write_text(json.dumps(final, indent=2))
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "epoch": config.epochs,
                "config": settings, "metrics": final,
                "recurrent_sigmas": {name: m.sigma for name, m in model.named_modules()
                                     if isinstance(m, axonal_recdel)}}, run_dir / "last.pth")
    diagnostics.plot_hidden_delay_evolution()
    plot_results(history, final, config, run_dir)
    print(f"Final training accuracy: {final['accuracy_percent']:.2f}%\nSaved: {run_dir}", flush=True)

    return history, final, run_dir

## Match the six models

The helpers below follow `experiments/compare_mem_ziyad.py`: seed once, generate
one shared reference with `generate_matched_parameters()`, then inject its weights,
biases and base delays into all six models. No complete master network is built.
After all copying, axonal and synaptic outputs are checked within each pathway on
a probe. Hybrids retain fixed random synaptic offsets, so their initial effective
delays differ. Models are built and trained in reference order: feedforward
axonal/synaptic/hybrid, then recurrent axonal/synaptic/hybrid.

Matching requires `kernel_count = 1`. All architecture flags are honored:
- `no_delay_in_first_layer = True`: remove input → first hidden DCLS delays.
- `no_delay_in_last_layer = True`: remove last hidden → output DCLS delays.
- `no_recurrence_in_last_layer = True`: remove the last hidden recurrence entirely.
  With one hidden layer, there are no recurrent connections left.
- `no_recurrent_delays = True`: use the vanilla RSNN with fixed zero delays on
  remaining recurrent connections. All three recurrent runs then use the same
  model, without hybrid offsets; their axonal/synaptic/hybrid labels are run slots.

With one hidden layer, enabling both `no_delay_*_layer` flags removes every
feedforward delay. With multiple hidden layers, intermediate projections retain
DCLS delays. Feedforward models always have no recurrence. The full reference is
still generated, so toggling flags preserves shared projection weights.
Models with no recurrent connections use the feedforward learning rates.

As in the reference, shared feedforward weights use `nn.Linear` default
initialization, irrespective of `init_ff_weights` / `init_dcls_weights`.

When updating an open notebook, rerun this helper cell before Run comparison.


In [ ]:
# Initialization helpers copied from experiments/compare_mem_ziyad.py.

MODELS = (
    ('Feedforward axonal',   'ff_axonal',      'SNN_axonal_feedforward_delays'),
    ('Feedforward synaptic', 'ff_synaptic',    'SNN_synaptic_feedforward_delays'),
    ('Feedforward hybrid',   'ff_hybrid',      'SNN_hybrid_feedforward_delays'),
    ('Recurrent axonal',     'rec_axonal',     'SNN_axonal_recurrent_delays'),
    ('Recurrent synaptic',   'rec_synaptic',   'SNN_synaptic_recurrent_delays'),
    ('Recurrent hybrid',     'rec_hybrid',     'SNN_recurrent_hybrid_delays'),
)

# axonal / synaptic init outputs must agree within each pathway group.
MATCHED_GROUPS = (
    ('feedforward', 'Feedforward axonal', 'Feedforward synaptic'),
    ('recurrent',   'Recurrent axonal',   'Recurrent synaptic'),
)

PARAM_COLORS = {'axonal': 'tab:blue', 'synaptic': 'tab:orange', 'hybrid': 'tab:green'}
# Pathway -> line style, so the six curves stay distinct (colour encodes the
# parametrization, style encodes the pathway).
PATHWAY_STYLES = {'Feedforward': '-', 'Recurrent': '--'}


def generate_matched_parameters(config):
    """Build one RNG-consistent set of feedforward + recurrent weights/delays.

    This is the shared reference every matched model is injected from: for each
    feedforward projection (one per hidden layer plus the output), a depthwise
    unit-weight delay filter (one learned axonal delay per source neuron, ``P``
    uniform-initialized and clamped like ``SNN_axonal_feedforward_delays``) followed
    by a plain ``nn.Linear`` projection; for each hidden layer, an ``axonal_recdel``
    recurrent layer (self-initializing its own weights/bias/delays). Mirrors what a
    both-pathways axonal model would build, without needing that network class.

    ``ff_w`` / ``ff_b`` / ``ff_delay`` (shape (in_channels, kernel_count)) are one
    entry per feedforward projection, in depth order; ``rec_*`` are one entry per
    recurrent layer (every hidden layer, never the output).
    """
    if config.kernel_count != 1:
        raise ValueError('matched_models requires kernel_count=1 (one delay per axon/synapse).')

    canon = {k: [] for k in ('ff_w', 'ff_b', 'ff_delay', 'rec_w', 'rec_b', 'rec_delay', 'rec_p_spread')}
    dim = config.input_size
    for idx, out_dim in enumerate(list(config.hidden_layers) + [config.output_size]):
        delay_config = deepcopy(config)
        delay_config.bias = False
        delay = dcls_module(delay_config, in_channels=dim, out_channels=dim, groups=dim)
        torch.nn.init.constant_(delay.weight, 1.0)
        torch.nn.init.uniform_(delay.P, a=config.init_pos_a, b=config.init_pos_b)
        delay.clamp_parameters()
        if config.DCLSversion == 'gauss':
            torch.nn.init.constant_(delay.SIG, config.siginit)
        canon['ff_delay'].append(delay.P.detach()[0, :, 0, :].clone())

        proj = torch.nn.Linear(dim, out_dim, bias=config.bias)
        canon['ff_w'].append(proj.weight.detach().clone())
        canon['ff_b'].append(proj.bias.detach().clone() if proj.bias is not None else None)

        if idx < len(config.hidden_layers):        # a recurrent layer follows every hidden layer
            rec = axonal_recdel(config, out_dim, config.neuron_module)
            canon['rec_w'].append(rec.recurrent_weights.detach().clone())
            canon['rec_b'].append(rec.recurrent_bias.detach().clone() if rec.use_rec_bias else None)
            canon['rec_delay'].append(rec.recurrent_delays.detach().clone())
            canon['rec_p_spread'].append(rec.p_spread.detach().clone() if rec.use_sig_p else None)
        dim = out_dim
    return canon


def _inject(model, canon):
    """Overwrite a model's shared components with the reference tensors."""
    fi = ri = 0
    with torch.no_grad():
        for m in model.layers:
            if isinstance(m, dcls_module) and not m.weight.requires_grad:
                # Depthwise delay filter: per-source delay, unit weights left frozen.
                # The projection's Linear follows and advances `fi`.
                learned_delay_parameter(m, 'P').copy_(
                    canon['ff_delay'][fi].view(1, m.in_channels, 1, m.kernel_count))
            elif isinstance(m, dcls_module):
                # Dense feedforward projection: DCLS weight carries the Linear
                # weight; every target shares the one per-source delay.
                m.weight.copy_(canon['ff_w'][fi].unsqueeze(-1).expand(-1, -1, m.kernel_count))
                if m.bias is not None and canon['ff_b'][fi] is not None:
                    m.bias.copy_(canon['ff_b'][fi])
                leaf = learned_delay_parameter(m, 'P')                # (1, O, in, k); O==1 for hybrid
                leaf.copy_(canon['ff_delay'][fi]
                           .view(1, 1, m.in_channels, m.kernel_count)
                           .expand(1, leaf.shape[1], m.in_channels, m.kernel_count))
                fi += 1
            elif isinstance(m, torch.nn.Linear):
                m.weight.copy_(canon['ff_w'][fi])
                if m.bias is not None and canon['ff_b'][fi] is not None:
                    m.bias.copy_(canon['ff_b'][fi])
                fi += 1
            elif isinstance(m, axonal_recdel):
                m.recurrent_weights.copy_(canon['rec_w'][ri])
                if getattr(m, 'use_rec_bias', False) and canon['rec_b'][ri] is not None:
                    m.recurrent_bias.copy_(canon['rec_b'][ri])
                leaf = learned_delay_parameter(m, 'recurrent_delays')  # (N,) axonal/hybrid, (N, N) synaptic
                base = canon['rec_delay'][ri]
                leaf.copy_(base if leaf.dim() == 1 else base[None, :].expand_as(leaf))
                if hasattr(m, 'p_spread') and canon['rec_p_spread'][ri] is not None:
                    m.p_spread.copy_(canon['rec_p_spread'][ri])
                ri += 1


def matched_models(config):
    """Build the six models, all seeded from one global reference."""
    config = deepcopy(config)
    # Generate the full reference even for disabled pathways: this keeps shared
    # weights identical across flag settings. _inject copies only existing modules;
    # Linear projections still advance fi, and only the final recurrence can be absent.

    seed_everything(config.seed)
    canon = generate_matched_parameters(config)

    built = {}
    for label, _slug, class_name in MODELS:
        cfg = deepcopy(config)
        cfg.model = ('SNN_vanilla_recurrent'
                     if label.startswith('Recurrent') and getattr(cfg, 'no_recurrent_delays', False)
                     else class_name)
        model = getattr(networks, cfg.model)(cfg)
        _inject(model, canon)
        if cfg.model == 'SNN_vanilla_recurrent':
            # Injection also copies delay tensors; restore the vanilla control.
            with torch.no_grad():
                for module in model.layers:
                    if isinstance(module, axonal_recdel):
                        module.recurrent_delays.zero_()
        set_epoch(model, cfg, 0)
        model.eval()
        built[label] = (cfg, model)

    with torch.no_grad():
        probe = torch.rand(config.time_window, 2, config.input_size,
                           generator=torch.Generator().manual_seed(123))
        for name, axonal_label, synaptic_label in MATCHED_GROUPS:
            out_axonal = built[axonal_label][1](probe)
            out_synaptic = built[synaptic_label][1](probe)
            torch.testing.assert_close(out_axonal, out_synaptic, atol=1e-5, rtol=1e-5,
                                       msg=f'{name}: axonal vs synaptic initial outputs differ')
            reset_states(built[axonal_label][1])
            reset_states(built[synaptic_label][1])

    return [built[label] for label, _slug, _class_name in MODELS]


def six_models(config):
    """Adapt the reference pairs to the notebook's labeled training/plotting API."""
    return [(label.title(), cfg, model)
            for (label, _slug, _class_name), (cfg, model)
            in zip(MODELS, matched_models(config), strict=True)]


def train_comparison_model(label, cfg, model, device, out, show_diagnostics=True):
    directory = out / label.lower().replace(' ', '_')
    history, final, _ = run(cfg, device, directory, model=model,
                            show_diagnostics=show_diagnostics)
    final['feedforward_delay_parameters'] = sum(learned_delay_parameter(m, 'P').numel() for m in model.layers if isinstance(m, dcls_module))
    final['recurrent_delay_parameters'] = sum(learned_delay_parameter(m, 'recurrent_delays').numel() for m in model.layers if isinstance(m, axonal_recdel))
    final['fixed_synaptic_offsets'] = sum(b.numel() for name, b in model.named_buffers() if name.endswith('.offsets'))
    model.cpu()
    return history, final


def train_pathway(config, family, device, out):
    # Build on CPU from the same reference and seed in each independent process.
    torch.set_num_threads(config.cpu_threads)
    models = six_models(config)
    records = {}
    for label, cfg, model in models:
        if label.startswith(family):
            cfg.assigned_device = config.assigned_device
            history, final = train_comparison_model(
                label, cfg, model, device, out, show_diagnostics=False)
            records[label] = {'history': history, 'final': final}
    return records


def run_parallel_groups(config, out, devices=('cuda:0', 'cuda:1')):
    """Fresh interpreters isolate CUDA, RNGs and plotting; no notebook fork."""
    import cloudpickle
    import time

    out = Path(out).resolve()
    out.mkdir(parents=True, exist_ok=True)
    # Serialize editable notebook functions/config, not live CUDA models or contexts.
    payload = out / 'worker_payload.pkl'
    payload.write_bytes(cloudpickle.dumps((train_pathway, config)))
    worker = out / 'pathway_worker.py'
    worker.write_text("""from pathlib import Path
import sys
import json
import cloudpickle
import torch

if __name__ == '__main__':
    payload, family, device, assigned_device, output = sys.argv[1:]
    if device.startswith('cuda'):
        torch.cuda.set_device(0)
    train_pathway, config = cloudpickle.loads(Path(payload).read_bytes())
    config.assigned_device = assigned_device
    records = train_pathway(config, family, torch.device(device), Path(output))
    (Path(output) / (family.lower() + '_results.json')).write_text(json.dumps(records))
""")
    jobs = []
    try:
        for family, assigned in zip(('Feedforward', 'Recurrent'), devices, strict=True):
            env = os.environ.copy()
            env['PYTHONPATH'] = os.pathsep.join([str(REPO_DIR / 'src'), env.get('PYTHONPATH', '')])
            env['MPLBACKEND'] = 'Agg'
            local_device = assigned
            if assigned.startswith('cuda:'):
                index = int(assigned.split(':')[1])
                visible = env.get('CUDA_VISIBLE_DEVICES')
                env['CUDA_VISIBLE_DEVICES'] = visible.split(',')[index].strip() if visible else str(index)
                local_device = 'cuda:0'  # Only the assigned physical GPU is visible.
            else:
                env['CUDA_VISIBLE_DEVICES'] = ''
            log_path = out / (family.lower() + '.log')
            writer = log_path.open('w')
            reader = log_path.open()
            try:
                process = subprocess.Popen(
                    [sys.executable, '-u', str(worker), str(payload), family,
                     local_device, assigned, str(out)], env=env,
                    stdout=writer, stderr=subprocess.STDOUT)
            except BaseException:
                writer.close()
                reader.close()
                raise
            jobs.append((family, process, writer, reader))
            print(f'{family} → {assigned}; log: {log_path}', flush=True)
        while True:
            for family, process, _, reader in jobs:
                for line in reader.readlines():
                    print(f'[{family}] {line}', end='', flush=True)
                code = process.poll()
                if code is not None and code != 0:
                    raise RuntimeError(f'{family} worker exited with code {code}; see {out / (family.lower() + ".log")}')
            if all(process.poll() is not None for _, process, _, _ in jobs):
                break
            time.sleep(0.5)
    finally:
        # Also stop the other worker if one fails or the notebook cell is interrupted.
        for _, process, _, _ in jobs:
            if process.poll() is None:
                process.terminate()
        for family, process, writer, reader in jobs:
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            for line in reader.readlines():
                print(f'[{family}] {line}', end='', flush=True)
            writer.close()
            reader.close()
    records = {}
    for family in ('feedforward', 'recurrent'):
        records.update(json.loads((out / (family + '_results.json')).read_text()))
    return records


## Run comparison

Run this cell again after editing and rerunning Configuration. It creates fresh models,
checks dataset equality, and saves the same per-model CSV, JSON, dataset, checkpoint
and summary plots as the script. With two GPUs, pathway groups train concurrently; each group trains its three models sequentially.
Before training, a Markdown table displays the resolved settings and per-model
initial learning rates. A copy is saved as `training_parameters.md` in the run folder.


In [ ]:
if DEVICE not in ('auto', 'cpu', 'cuda'):
    raise ValueError('DEVICE must be auto, cpu, or cuda')
if min(config.epochs, config.num_samples, config.batch_size, config.cpu_threads,
       config.input_size, config.output_size, config.time_window, *config.hidden_layers) < 1:
    raise ValueError('Epochs, samples, batch size, CPU threads and dimensions must be positive')
if config.task_type not in ('temporal', 'spatial') or config.readout not in ('mean', 'sum', 'last'):
    raise ValueError('Invalid task_type or readout')
device = torch.device(('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE == 'auto' else DEVICE)
if device.type == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA requested but unavailable; enable a GPU accelerator or use DEVICE="auto"')
if PARALLEL_GPUS not in ('auto', True, False):
    raise ValueError('PARALLEL_GPUS must be auto, True, or False')
use_two_gpus = device.type == 'cuda' and torch.cuda.device_count() >= 2
if PARALLEL_GPUS is True and not use_two_gpus:
    raise RuntimeError('PARALLEL_GPUS=True requires DEVICE=auto/cuda and two visible CUDA GPUs')
use_two_gpus = use_two_gpus and PARALLEL_GPUS is not False
torch.set_num_threads(config.cpu_threads)
# Keep the settings associated with these results even if Configuration is rerun later.
run_config = deepcopy(config)
out = OUTPUT_ROOT / f'{config.task_type}_seed{config.seed}_{datetime.now():%Y-%m-%d-%H-%M-%S-%f}'
models = six_models(run_config)
assignments = {label: ('cuda:0' if label.startswith('Feedforward') else 'cuda:1')
               if use_two_gpus else str(device) for label, _, _ in models}
for label, cfg, _ in models:
    cfg.assigned_device = assignments[label]
recap_device = 'Feedforward: cuda:0; Recurrent: cuda:1' if use_two_gpus else device
display(Markdown(save_training_recap(models, recap_device, out)))
(out / 'execution.json').write_text(json.dumps({'parallel': use_two_gpus, 'devices': assignments}, indent=2))
histories, results = {}, {}
if use_two_gpus:
    # Workers rebuild the matched CPU initialization using these notebook helpers.
    records = run_parallel_groups(run_config, out)
    for label, _, _ in models:
        histories[label], results[label] = records[label]['history'], records[label]['final']
else:
    for label, cfg, model in models:
        histories[label], results[label] = train_comparison_model(label, cfg, model, device, out)

# Check all six saved datasets after both workers finish, before reporting success.
reference_data = None
for label in results:
    data = torch.load(out / label.lower().replace(' ', '_') / 'dataset.pt', weights_only=True)
    if reference_data is None:
        reference_data = data
    else:
        assert all(torch.equal(data[k], reference_data[k]) for k in data)

(out / 'comparison.json').write_text(json.dumps({
    'initialization': 'Initialization follows compare_mem_ziyad.py: generate_matched_parameters creates one shared reference for feedforward weights/biases and base delays, and recurrent weights/biases and base delays. All six models receive the applicable reference tensors. Axonal/synaptic initial outputs are checked within each pathway after injection; hybrids retain fixed random synaptic offsets on enabled delay pathways. Architecture flags are honored; no_recurrent_delays uses identical vanilla RSNNs with fixed zero delays for recurrent runs. Recurrent models have no feedforward delays; feedforward models have no recurrence. Same dataset and batch order for all six models.',
    'results': results,
}, indent=2))
print(f'Six-model comparison saved: {out}', flush=True)

## Plot comparison

This cell can be rerun without training. Solid curves show recurrent delays;
dashed curves and hatched bars show feedforward delays. The PNG/PDF comparison,
`comparison.json` and per-model folders are saved under the printed output path
(in `/kaggle/working/exp/MEM/delay_location_comparison/` on Kaggle).

In [ ]:
def plot_comparison(histories, results, config, out):
    fig = plt.figure(figsize=(16, 9), layout='constrained')
    grid = fig.add_gridspec(2, 2)
    loss_ax, acc_ax, bar_ax = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[1, :])
    colors = {'Axonal': 'tab:blue', 'Synaptic': 'tab:orange', 'Hybrid': 'tab:green'}
    for label, history in histories.items():
        family, kind = label.split()
        style = '-' if family == 'Recurrent' else '--'
        epochs = [r['epoch'] for r in history]
        loss_ax.plot(epochs, [r['loss'] for r in history], color=colors[kind], linestyle=style, label=label)
        acc_ax.plot(epochs, [r['accuracy_percent'] for r in history], color=colors[kind], linestyle=style, label=label)
    loss_ax.set(xlabel='Epoch', ylabel='Cross-entropy loss', title='Training loss')
    acc_ax.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Training accuracy', ylim=(0, 105))
    for axis in (loss_ax, acc_ax):
        axis.legend(fontsize=9, ncol=2)
        axis.grid(alpha=0.25)
    labels = [f"{label.replace(' ', chr(10), 1)}\n{r['trainable_parameters']:,} parameters" for label, r in results.items()]
    bars = bar_ax.bar(labels, [r['accuracy_percent'] for r in results.values()],
                      color=[colors[label.split()[1]] for label in results])
    for bar, label in zip(bars, results):
        if label.startswith('Feedforward '):
            bar.set_hatch('//')
    bar_ax.bar_label(bars, labels=[f"{r['accuracy_percent']:.2f}%" for r in results.values()], padding=4)
    bar_ax.set(ylabel='Accuracy (%)', title='Final training accuracy', ylim=(0, 110))
    topology = ' → '.join(map(str, [config.input_size] + config.hidden_layers + [config.output_size]))
    if getattr(config, 'no_recurrent_delays', False):
        loss_ax.set_title('Training loss (recurrent runs: identical zero-delay RSNNs)')
    fig.suptitle(f'Recurrent-only vs feedforward-only delays | {config.task_type}, {config.num_samples} samples, topology {topology}\n'
                 'Solid: recurrent delays only · Dashed: feedforward delays only')
    for extension in ('png', 'pdf'):
        fig.savefig(out / f'delay_location_comparison.{extension}', dpi=180)
    plt.show()
    plt.close(fig)


plot_comparison(histories, results, run_config, out)